In [1]:
from benchmark import ModelBenchmark
from helpers import DATA_PATH, get_data_from_file

In [2]:
corrupt, clean = get_data_from_file('small')

### Baseline

In [203]:
from symspellpy import symspellpy
import pkg_resources

max_edit_distance = 2
prefix_length = 7

sym_spell = symspellpy.SymSpell(max_edit_distance, prefix_length)
dictionary_path = pkg_resources.resource_filename(
        "symspellpy", "frequency_dictionary_en_82_765.txt")
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)
# bigram_path = pkg_resources.resource_filename(
#     "symspellpy", "frequency_bigramdictionary_en_243_342.txt")
# sym_spell.load_bigram_dictionary(bigram_path, term_index=0, count_index=2)

sym_spell.create_dictionary(DATA_PATH + "corpus.txt", encoding='utf-8')
input_phrase = corrupt[0]
suggestions = sym_spell.lookup_compound(input_phrase, max_edit_distance=max_edit_distance)



In [211]:
print(suggestions[0].term)

team number tr 1 p span contents 0


In [15]:
benchmark = ModelBenchmark(device='cpu')

In [ ]:
benchmark.benchmark_model(sym_spell,
                          clean,
                          corrupt,
                          "symspell",
                          lambda model, data: model.correct_string(data),
                          warm_up_runs=0,
                          num_runs=2)


### Neuspell


In [4]:
from neuspell import BertChecker

checker = BertChecker(device='cuda')
checker.from_pretrained()

data folder is set to `C:\FIT\bakalarka\.venv\Lib\site-packages\neuspell\../neuspell_data` script
loading vocab from path:C:\FIT\bakalarka\.venv\Lib\site-packages\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise\vocab.pkl
initializing model
loading pretrained weights from path:C:\FIT\bakalarka\.venv\Lib\site-packages\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise
Loading model params from checkpoint dir: C:\FIT\bakalarka\.venv\Lib\site-packages\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise


#### Bench

In [6]:
benchmark = ModelBenchmark()

In [7]:
benchmark.benchmark_model(checker,
                          corrupt,
                          clean,
                          "neuspell-bert",
                          lambda model, data: model.correct_string(data),
                          warm_up_runs=0,
                          num_runs=2)


Starting 0 warm-up iterations for neuspell-bert...
Finished warm-up after 0.0 seconds.
Starting benchmark iterations...
Finished 1/2 iteration in 2.312450647354126 seconds.
Finished 2/2 iteration in 2.131199359893799 seconds.


Benchmark results:
	Model: neuspell-bert
	Size: 706.534797668457 MB
	Inference Time: 2.2218250036239624 s
	Peak Memory: 0.040357112884521484 MB
	GPU Memory: 841.85302734375 MB
	Throughput: 349.39364587594406 tokens/sec
	Throughput: 28.853152691690866 sentences/sec
	Accuracy tokens: 20.129%
	Accuracy sentences: 0.000%
	Correct → Correct: 144.0
	Correct → Incorrect: 452.0
	Incorrect → Correct: 12.0
	Incorrect → Incorrect: 167.0
	Word Correction Rate: 6.704%
	Word Incorrection Rate: 75.839%

In [56]:
corrupt[0]

'team_number = tr[1].p.span.contents[0]'

In [57]:
checker.correct_string(corrupt[0])
# checker.correct_string(" I luk forawd to itd.")

'team _ number = tr [ 1 ] . p . span . contents [ 0 ]'

In [11]:
clean[1]

'* The internal method that handles the pointer over event from the browser.'

### FIX

In [5]:
from transformers import BertTokenizerFast
import numpy as np

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
test_string = corrupt[0]
test_string_clean = clean[0]
tokens_mapping = tokenizer(test_string, return_offsets_mapping=True)
original_offsets = tokens_mapping['offset_mapping']
transformed_tokens = checker.correct_string(test_string)
print(f"input: {test_string}", f"output: {transformed_tokens}", sep="\n")

input: team_number = tr[1].p.span.contents[0]
output: team _ number = tr [ 1 ] . p . span . contents [ 0 ]


In [6]:
print(*original_offsets[1:-1])

(0, 4) (4, 5) (5, 11) (12, 13) (14, 15) (15, 16) (16, 17) (17, 18) (18, 19) (19, 20) (20, 21) (21, 22) (22, 26) (26, 27) (27, 35) (35, 36) (36, 37) (37, 38)


In [7]:
print(tokenizer.tokenize(test_string))
print(len(tokenizer.tokenize(test_string)))
print(len(transformed_tokens.split()))

['team', '_', 'number', '=', 't', '##r', '[', '1', ']', '.', 'p', '.', 'span', '.', 'contents', '[', '0', ']']
18
17


In [8]:
pretok_sent = []
offsets_merged = []
for token, offset in zip(tokenizer.tokenize(transformed_tokens), original_offsets[1:-1]):
    if token.startswith("##"):
        pretok_sent[-1] = pretok_sent[-1] + token[2:]
        offsets_merged[-1] = (offsets_merged[-1][0], offset[1])
    else:
        pretok_sent.append(token)
        offsets_merged.append(offset)
print(pretok_sent)
print(offsets_merged)

['team', '_', 'number', '=', 'tr', '[', '1', ']', '.', 'p', '.', 'span', '.', 'contents', '[', '0', ']']
[(0, 4), (4, 5), (5, 11), (12, 13), (14, 16), (16, 17), (17, 18), (18, 19), (19, 20), (20, 21), (21, 22), (22, 26), (26, 27), (27, 35), (35, 36), (36, 37), (37, 38)]


In [10]:
# first and last offsets are [CLS] and [SEP]
offsets = np.array(offsets_merged)
# boolean mask checking if the offsets align or next start index is greater than previous end
# indicating space in the original text
mask = offsets[1:, 0] > offsets[:-1, 1]
mask = np.append(mask, False)
print(mask)
print(mask.shape)
tokens_arr = np.array(pretok_sent)
# add spaces back  at corresponding places
tokens_with_space = np.where(mask, tokens_arr + " ", tokens_arr)
reconstructed_text = "".join(tokens_with_space)
reconstructed_text

[False False  True  True False False False False False False False False
 False False False False False]
(17,)


'team_number = tr[1].p.span.contents[0]'

### T5

In [11]:
from happytransformer import HappyTextToText

# Load T5 model for grammar/spelling correction
happy_tt = HappyTextToText("T5", "vennify/t5-base-grammar-correction")

# Example sentence with typos
input_text = corrupt[0]

# Correct the sentence
output = happy_tt.generate_text(f"grammar: {input_text}")
print(output.text, clean[0], sep='\n')


04/01/2025 15:56:13 - INFO - happytransformer.happy_transformer -   Using device: cuda:0
04/01/2025 15:56:14 - INFO - happytransformer.happy_transformer -   Moving model to cuda:0
04/01/2025 15:56:14 - INFO - happytransformer.happy_transformer -   Initializing a pipeline
Device set to use cuda:0


Team_number = tr[1].p.span.contents[0]
team_number = tds[1].p.span.contents[0]
